<a href="https://colab.research.google.com/github/kfklaihk/Works/blob/main/Parse_Find_Pattern_Con_Where.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
#!/usr/bin/env python3
"""
Oracle SQL Relationship Analyzer - Incremental Processing
Processes one file at a time and saves results immediately to prevent data loss.
"""

import zipfile
import pandas as pd
import re
import os
import json
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import List, Dict, Optional
import csv
from collections import defaultdict
from google.colab import files
import io
from datetime import datetime

@dataclass
class SQLFinding:
    """Class to store SQL relationship findings"""
    source_table: str
    source_field: str
    file_path: str
    relationship_type: str
    sql_snippet: str
    destination_table: str
    destination_field: str
    condition: str
    line_number: int

    def to_dict(self):
        return {
            'Source Table': self.source_table,
            'Source Field': self.source_field,
            'File Path': self.file_path,
            'Relationship Type': self.relationship_type,
            'Destination Table': self.destination_table,
            'Destination Field': self.destination_field,
            'Condition/Where Clause': self.condition,
            'Line Number': self.line_number,
            'SQL Snippet': self.sql_snippet
        }

class IncrementalSQLAnalyzer:
    def __init__(self):
        self.processed_files_log = "processed_files.json"
        self.results_file = "sql_findings_partial.csv"
        self.checkpoint_file = "analysis_checkpoint.json"

    def safe_upper(self, value) -> str:
        """Safely convert to uppercase, handling None"""
        if value is None:
            return ""
        return str(value).upper()

    def safe_string(self, value) -> str:
        """Safely convert to string, handling None"""
        if value is None:
            return ""
        return str(value)

    def load_checkpoint(self) -> Dict:
        """Load processing checkpoint to resume if needed"""
        if os.path.exists(self.checkpoint_file):
            with open(self.checkpoint_file, 'r') as f:
                return json.load(f)
        return {"processed_files": [], "total_findings": 0}

    def save_checkpoint(self, checkpoint: Dict):
        """Save current processing state"""
        with open(self.checkpoint_file, 'w') as f:
            json.dump(checkpoint, f, indent=2)
        print(f"Checkpoint saved: {checkpoint['processed_files'][-1] if checkpoint['processed_files'] else 'None'}")

    def append_findings_to_csv(self, findings: List[SQLFinding]):
        """Append findings to CSV immediately (don't wait for all files)"""
        if not findings:
            return

        data = [f.to_dict() for f in findings]
        df = pd.DataFrame(data)

        # Append mode - create if not exists, append with header only if new
        file_exists = os.path.exists(self.results_file)
        mode = 'a' if file_exists else 'w'
        header = not file_exists

        df.to_csv(self.results_file, mode=mode, header=header, index=False, quoting=csv.QUOTE_ALL)
        print(f"  -> Saved {len(findings)} findings to {self.results_file}")

    def upload_files(self):
        """Upload both required files"""
        print("Step 1: Upload the ZIP file containing SQL files...")
        uploaded_zip = files.upload()
        zip_filename = list(uploaded_zip.keys())[0]

        print("\nStep 2: Upload the Excel file with requirements...")
        uploaded_excel = files.upload()
        excel_filename = list(uploaded_excel.keys())[0]

        return zip_filename, excel_filename

    def read_requirements(self, excel_path: str) -> pd.DataFrame:
        """Read and filter Excel requirements (Column C > 3)"""
        print(f"\nReading Excel: {excel_path}")
        df = pd.read_excel(excel_path, sheet_name="Export Worksheet")

        # Filter where Column C (index 2) > 3
        df_filtered = df[df.iloc[:, 2] > 3].copy()
        df_filtered.columns = ['source_table', 'source_field', 'threshold'] + list(df_filtered.columns[3:])

        # Clean up - remove None/NaN values
        df_filtered = df_filtered.dropna(subset=['source_table', 'source_field'])

        print(f"Total rows in Excel: {len(df)}")
        print(f"Rows with Column C > 3: {len(df_filtered)}")
        print(f"Requirements to check: {len(df_filtered)}")

        return df_filtered[['source_table', 'source_field']]

    def get_sql_files_list(self, zip_path: str) -> List[str]:
        """Get list of SQL files from ZIP without extracting all at once"""
        sql_files = []
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            for file_info in zip_ref.filelist:
                if file_info.filename.endswith('.sql'):
                    sql_files.append(file_info.filename)
        return sorted(sql_files)

    def extract_single_sql(self, zip_path: str, file_name: str) -> str:
        """Extract and return content of a single SQL file"""
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            content = zip_ref.read(file_name).decode('utf-8', errors='ignore')
        return content

    def normalize_sql(self, sql: str) -> str:
        """Normalize SQL for parsing"""
        if not sql:
            return ""
        sql = re.sub(r'--.*$', '', sql, flags=re.MULTILINE)
        sql = re.sub(r'/\*.*?\*/', '', sql, flags=re.DOTALL)
        sql = ' '.join(sql.split())
        return sql.upper()

    def extract_table_aliases(self, sql: str) -> Dict[str, str]:
        """Extract table aliases"""
        aliases = {}
        if not sql:
            return aliases

        patterns = [
            r'FROM\s+(\w+)(?:\s+(?:AS\s+)?(\w+))?',
            r'JOIN\s+(\w+)(?:\s+(?:AS\s+)?(\w+))?',
        ]
        for pattern in patterns:
            matches = re.finditer(pattern, sql, re.IGNORECASE)
            for match in matches:
                table = match.group(1)
                alias = match.group(2) if match.group(2) else table
                if table and alias:
                    aliases[alias.upper()] = table.upper()
                    aliases[table.upper()] = table.upper()
        return aliases

    def get_derived_names(self, field_name: str) -> List[str]:
        """Generate possible derived field names"""
        if not field_name:
            return []

        variations = [field_name.upper()]
        prefixes = ['NEW_', 'OLD_', 'CALC_', 'V_', 'VAR_']
        suffixes = ['_NEW', '_CALC', '_VAL', '_ID']

        for prefix in prefixes:
            variations.append(f"{prefix}{field_name.upper()}")
        for suffix in suffixes:
            variations.append(f"{field_name.upper()}{suffix}")

        return variations

    def find_derived_variables(self, sql: str, source_table: str, source_field: str) -> List[str]:
        """Find variables derived from source field"""
        derived = []
        if not sql or not source_field:
            return derived

        upper_sql = sql.upper()
        upper_table = source_table.upper() if source_table else ""
        upper_field = source_field.upper()

        # Pattern: variable := source_table.source_field or variable := source_field
        if upper_table:
            pattern = rf'(\w+)\s*:=\s*(?:{re.escape(upper_table)}\.)?{re.escape(upper_field)}\b'
        else:
            pattern = rf'(\w+)\s*:=\s*{re.escape(upper_field)}\b'

        matches = re.finditer(pattern, upper_sql)
        for match in matches:
            var = match.group(1)
            if var:
                derived.append(var)

        # Pattern: variable = calculation involving source_field
        pattern = rf'(\w+)\s*[=:]\s*[^;]*?\b{re.escape(upper_field)}\b'
        matches = re.finditer(pattern, upper_sql)
        for match in matches:
            var = match.group(1)
            if var and var not in derived:
                derived.append(var)

        return derived

    def parse_insert_select(self, sql: str) -> List[Dict]:
        """Parse INSERT...SELECT statements"""
        results = []
        if not sql:
            return results

        pattern = r'INSERT\s+(?:INTO\s+)?(\w+)\s*\(([^)]+)\)\s*SELECT\s+([^}]+?)(?:\s+FROM\s+(\w+)(?:\s+AS\s+(\w+))?)?(?:\s+WHERE\s+([^;]+))?'

        matches = re.finditer(pattern, sql, re.IGNORECASE | re.DOTALL)
        for match in matches:
            results.append({
                'type': 'INSERT_SELECT',
                'dest_table': match.group(1) or "",
                'dest_fields': [f.strip() for f in (match.group(2) or "").split(',') if f.strip()],
                'select_clause': match.group(3) or "",
                'from_table': match.group(4) or "",
                'from_alias': match.group(5) or "",
                'where_clause': match.group(6) if match.group(6) else "",
                'full_match': match.group(0) or ""
            })
        return results

    def parse_insert_values(self, sql: str) -> List[Dict]:
        """Parse INSERT...VALUES statements"""
        results = []
        if not sql:
            return results

        pattern = r'INSERT\s+(?:INTO\s+)?(\w+)\s*\(([^)]+)\)\s*VALUES\s*\(([^)]+)\)(?:\s+WHERE\s+([^;]+))?'

        matches = re.finditer(pattern, sql, re.IGNORECASE | re.DOTALL)
        for match in matches:
            results.append({
                'type': 'INSERT_VALUES',
                'dest_table': match.group(1) or "",
                'dest_fields': [f.strip() for f in (match.group(2) or "").split(',') if f.strip()],
                'values': match.group(3) or "",
                'where_clause': match.group(4) if match.group(4) else "",
                'full_match': match.group(0) or ""
            })
        return results

    def parse_cursor_definitions(self, sql: str) -> List[Dict]:
        """Parse CURSOR definitions with UNION"""
        results = []
        if not sql:
            return results

        pattern = r'CURSOR\s+(\w+)\s+IS\s+(SELECT\s+.+?(?:UNION\s+ALL?\s+SELECT\s+.+?)+)'

        matches = re.finditer(pattern, sql, re.IGNORECASE | re.DOTALL)
        for match in matches:
            cursor_name = match.group(1) or ""
            union_sql = match.group(2) or ""

            select_parts = re.split(r'UNION\s+ALL?\s+', union_sql, flags=re.IGNORECASE)
            selects = []
            for part in select_parts:
                from_match = re.search(r'FROM\s+(\w+)(?:\s+AS\s+(\w+))?', part, re.IGNORECASE)
                if from_match:
                    selects.append({
                        'from_table': from_match.group(1) or "",
                        'from_alias': from_match.group(2) or "",
                        'select_clause': part or ""
                    })

            results.append({
                'type': 'CURSOR_UNION',
                'cursor_name': cursor_name,
                'selects': selects,
                'full_match': match.group(0) or ""
            })
        return results

    def get_line_number(self, full_sql: str, snippet: str) -> int:
        """Get line number of snippet"""
        if not full_sql or not snippet:
            return 0
        try:
            search_text = snippet[:50] if len(snippet) > 50 else snippet
            pos = full_sql.index(search_text)
            return full_sql[:pos].count('\n') + 1
        except ValueError:
            return 0

    def analyze_single_file(self, file_path: str, sql: str, requirements: pd.DataFrame) -> List[SQLFinding]:
        """Analyze a single SQL file against all requirements"""
        findings = []

        if not sql or requirements.empty:
            return findings

        normalized_sql = self.normalize_sql(sql)

        print(f"  Checking {len(requirements)} requirements...")

        for _, row in requirements.iterrows():
            source_table = self.safe_upper(row.get('source_table'))
            source_field = self.safe_upper(row.get('source_field'))

            if not source_table or not source_field:
                continue

            # Quick check: is source table mentioned?
            aliases = self.extract_table_aliases(sql)
            table_found = (source_table in normalized_sql or
                          source_table in aliases.values())

            if not table_found:
                continue

            # Parse statements
            try:
                insert_selects = self.parse_insert_select(sql)
                insert_values = self.parse_insert_values(sql)
                cursors = self.parse_cursor_definitions(sql)
            except Exception as e:
                print(f"    Warning: Error parsing SQL patterns: {e}")
                continue

            # Check Pattern 1: Multiple INSERT...SELECT with different conditions
            try:
                findings.extend(self._check_insert_select_pattern(
                    file_path, sql, source_table, source_field,
                    insert_selects, cursors
                ))
            except Exception as e:
                print(f"    Warning: Error in INSERT SELECT check: {e}")

            # Check Pattern 2: INSERT...VALUES vs INSERT...SELECT
            try:
                findings.extend(self._check_insert_values_pattern(
                    file_path, sql, source_table, source_field,
                    insert_values, insert_selects
                ))
            except Exception as e:
                print(f"    Warning: Error in INSERT VALUES check: {e}")

        return findings

    def _check_insert_select_pattern(self, file_path: str, sql: str, source_table: str,
                                    source_field: str, insert_selects: List[Dict],
                                    cursors: List[Dict]) -> List[SQLFinding]:
        """Check for multiple INSERT...SELECT from same source"""
        findings = []
        source_groups = defaultdict(list)

        # Process direct INSERT...SELECT
        for ins in insert_selects:
            from_table = self.safe_upper(ins.get('from_table'))
            aliases = self.extract_table_aliases(sql)

            if from_table == source_table or aliases.get(from_table) == source_table:
                select_clause = self.safe_upper(ins.get('select_clause'))
                derived_names = self.get_derived_names(source_field)

                if source_field in select_clause or any(d in select_clause for d in derived_names):
                    source_groups[from_table].append({
                        'type': 'INSERT_SELECT',
                        'data': ins,
                        'cursor': None
                    })

        # Process CURSOR-based INSERT...SELECT
        for cursor in cursors:
            cursor_name = cursor.get('cursor_name', '')
            if not cursor_name:
                continue

            cursor_pattern = rf'INSERT\s+(?:INTO\s+)?(\w+).*?SELECT.*?FROM\s+{re.escape(cursor_name)}'
            cursor_inserts = re.finditer(cursor_pattern, sql, re.IGNORECASE | re.DOTALL)

            for ins_match in cursor_inserts:
                for select in cursor.get('selects', []):
                    from_table = self.safe_upper(select.get('from_table'))
                    if from_table == source_table:
                        select_clause = self.safe_upper(select.get('select_clause'))
                        if source_field in select_clause:
                            where_match = re.search(r'WHERE\s+(.+?)(?:;|$)', ins_match.group(0), re.IGNORECASE)
                            where_clause = where_match.group(1) if where_match else ""

                            # Extract destination table from INSERT
                            dest_match = re.search(r'INSERT\s+(?:INTO\s+)?(\w+)', ins_match.group(0), re.IGNORECASE)
                            dest_table = dest_match.group(1) if dest_match else "UNKNOWN"

                            source_groups[from_table].append({
                                'type': 'CURSOR_INSERT',
                                'data': ins_match.group(0),
                                'cursor': cursor_name,
                                'where_clause': where_clause,
                                'dest_table': dest_table
                            })

        # Check for 2+ different conditions
        for src, operations in source_groups.items():
            if len(operations) >= 2:
                conditions = []
                for op in operations:
                    if op['type'] == 'INSERT_SELECT':
                        cond = op['data'].get('where_clause', '')
                    else:
                        cond = op.get('where_clause', '')
                    conditions.append(cond)

                if len(set(conditions)) > 1:
                    for i, op in enumerate(operations):
                        if op['type'] == 'INSERT_SELECT':
                            data = op['data']
                            findings.append(SQLFinding(
                                source_table=source_table,
                                source_field=source_field,
                                file_path=file_path,
                                relationship_type='MULTI_INSERT_SELECT_DIFFERENT_CONDITIONS',
                                sql_snippet=self.safe_string(data.get('full_match'))[:500],
                                destination_table=self.safe_string(data.get('dest_table')),
                                destination_field=', '.join(data.get('dest_fields', [])),
                                condition=conditions[i],
                                line_number=self.get_line_number(sql, self.safe_string(data.get('full_match')))
                            ))
                        else:
                            findings.append(SQLFinding(
                                source_table=source_table,
                                source_field=source_field,
                                file_path=file_path,
                                relationship_type='CURSOR_UNION_INSERT',
                                sql_snippet=self.safe_string(op.get('data'))[:500],
                                destination_table=self.safe_string(op.get('dest_table', 'UNKNOWN')),
                                destination_field='UNKNOWN',
                                condition=conditions[i],
                                line_number=self.get_line_number(sql, self.safe_string(op.get('data')))
                            ))

        return findings

    def _check_insert_values_pattern(self, file_path: str, sql: str, source_table: str,
                                    source_field: str, insert_values: List[Dict],
                                    insert_selects: List[Dict]) -> List[SQLFinding]:
        """Check for INSERT...VALUES vs INSERT...SELECT"""
        findings = []
        derived_vars = self.find_derived_variables(sql, source_table, source_field)

        for iv in insert_values:
            values_clause = self.safe_upper(iv.get('values'))
            dest_table_iv = self.safe_upper(iv.get('dest_table'))

            for var in derived_vars:
                if var and var.upper() in values_clause:
                    for ins in insert_selects:
                        dest_table_ins = self.safe_upper(ins.get('dest_table'))
                        if dest_table_ins == dest_table_iv:
                            iv_fields = set(f.upper() for f in iv.get('dest_fields', []) if f)
                            ins_fields = set(f.upper() for f in ins.get('dest_fields', []) if f)
                            common_fields = iv_fields & ins_fields

                            if common_fields:
                                findings.append(SQLFinding(
                                    source_table=source_table,
                                    source_field=source_field,
                                    file_path=file_path,
                                    relationship_type='INSERT_VALUES_VS_SELECT',
                                    sql_snippet=f"VALUES: {self.safe_string(iv.get('full_match'))[:300]}... | SELECT: {self.safe_string(ins.get('full_match'))[:300]}",
                                    destination_table=iv.get('dest_table') or "",
                                    destination_field=', '.join(common_fields),
                                    condition=f"VALUES_WHERE: {iv.get('where_clause') or 'N/A'}",
                                    line_number=self.get_line_number(sql, self.safe_string(iv.get('full_match')))
                                ))
        return findings

    def process_all_files(self, zip_path: str, requirements: pd.DataFrame):
        """Process SQL files one by one with checkpointing"""
        checkpoint = self.load_checkpoint()
        processed_files = set(checkpoint.get('processed_files', []))

        # Get list of all SQL files
        all_sql_files = self.get_sql_files_list(zip_path)
        total_files = len(all_sql_files)

        print(f"\nTotal SQL files in ZIP: {total_files}")
        print(f"Already processed: {len(processed_files)}")
        print(f"Remaining: {total_files - len(processed_files)}")

        files_to_process = [f for f in all_sql_files if f not in processed_files]

        for idx, file_name in enumerate(files_to_process, 1):
            print(f"\n[{idx}/{len(files_to_process)}] Processing: {file_name}")

            try:
                # Extract and analyze single file
                sql_content = self.extract_single_sql(zip_path, file_name)

                # Skip empty files
                if not sql_content or not sql_content.strip():
                    print(f"  ⚠ Skipping empty file: {file_name}")
                    checkpoint['processed_files'].append(file_name)
                    checkpoint['last_processed'] = file_name
                    checkpoint['timestamp'] = datetime.now().isoformat()
                    self.save_checkpoint(checkpoint)
                    continue

                # Analyze against all requirements
                findings = self.analyze_single_file(file_name, sql_content, requirements)

                # Save findings immediately
                if findings:
                    self.append_findings_to_csv(findings)
                    checkpoint['total_findings'] = checkpoint.get('total_findings', 0) + len(findings)

                # Mark as processed and save checkpoint
                checkpoint['processed_files'].append(file_name)
                checkpoint['last_processed'] = file_name
                checkpoint['timestamp'] = datetime.now().isoformat()
                self.save_checkpoint(checkpoint)

                print(f"  ✓ Completed: {file_name} ({len(findings)} findings)")

            except Exception as e:
                print(f"  ✗ Error processing {file_name}: {str(e)}")
                # Save error to checkpoint but continue
                checkpoint['errors'] = checkpoint.get('errors', [])
                checkpoint['errors'].append({'file': file_name, 'error': str(e)})
                self.save_checkpoint(checkpoint)
                continue

        print(f"\n{'='*60}")
        print(f"PROCESSING COMPLETE")
        print(f"Total files processed: {len(checkpoint.get('processed_files', []))}")
        print(f"Total findings: {checkpoint.get('total_findings', 0)}")
        print(f"Results saved to: {self.results_file}")

        return checkpoint

    def finalize_and_download(self):
        """Download the final results"""
        print(f"\nPreparing final download...")

        if os.path.exists(self.results_file):
            files.download(self.results_file)
            print(f"Downloaded: {self.results_file}")
        else:
            print("No results file found!")

        # Also download checkpoint for reference
        if os.path.exists(self.checkpoint_file):
            files.download(self.checkpoint_file)


def main():
    """Main execution with incremental processing"""
    analyzer = IncrementalSQLAnalyzer()

    # Upload files
    zip_path, excel_path = analyzer.upload_files()

    # Read requirements (once)
    requirements = analyzer.read_requirements(excel_path)

    # Process all files incrementally
    analyzer.process_all_files(zip_path, requirements)

    # Final download
    analyzer.finalize_and_download()

    print("\n✓ Analysis complete! Check your downloads.")


# Run in Google Colab
if __name__ == "__main__":
    main()

Step 1: Upload the ZIP file containing SQL files...


Saving CtrlM.zip to CtrlM (1).zip

Step 2: Upload the Excel file with requirements...


Saving Source_To_Multi_Targets.xlsx to Source_To_Multi_Targets (1).xlsx

Reading Excel: Source_To_Multi_Targets (1).xlsx
Total rows in Excel: 971
Rows with Column C > 3: 405
Requirements to check: 405

Total SQL files in ZIP: 183
Already processed: 0
Remaining: 183

[1/183] Processing: CtrlM/CtrlM/CtrlM-part0-Filelist/0 - Index and Constraint.sql
  Checking 405 requirements...


/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Checkpoint saved: CtrlM/CtrlM/CtrlM-part0-Filelist/0 - Index and Constraint.sql
  ✓ Completed: CtrlM/CtrlM/CtrlM-part0-Filelist/0 - Index and Constraint.sql (0 findings)

[2/183] Processing: CtrlM/CtrlM/CtrlM-part0-Filelist/0. DMDeleteSchemaStat.sql
  Checking 405 requirements...
Checkpoint saved: CtrlM/CtrlM/CtrlM-part0-Filelist/0. DMDeleteSchemaStat.sql
  ✓ Completed: CtrlM/CtrlM/CtrlM-part0-Filelist/0. DMDeleteSchemaStat.sql (0 findings)

[3/183] Processing: CtrlM/CtrlM/CtrlM-part0-Filelist/1. TRUNCATE Tables_Joe_v3.sql
  Checking 405 requirements...
Checkpoint saved: CtrlM/CtrlM/CtrlM-part0-Filelist/1. TRUNCATE Tables_Joe_v3.sql
  ✓ Completed: CtrlM/CtrlM/CtrlM-part0-Filelist/1. TRUNCATE Tables_Joe_v3.sql (0 findings)

[4/183] Processing: CtrlM/CtrlM/CtrlM-part0-Filelist/1.0.5a - Drop_Define_Tables.sql
  Checking 405 requirements...
Checkpoint saved: CtrlM/CtrlM/CtrlM-part0-Filelist/1.0.5a - Drop_Define_Tables.sql
  ✓ Completed: CtrlM/CtrlM/CtrlM-part0-Filelist/1.0.5a - Drop_Define

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✓ Analysis complete! Check your downloads.
